# Jetson Nano NMS Kernel — Compile, Test, and Benchmark on T4

This notebook compiles the Jetson Nano CUDA NMS kernel (targeting sm_53, but compiled for the T4's native arch here for testing), verifies correctness against `torchvision.ops.nms`, and benchmarks latency.

**Runtime requirement:** GPU runtime (T4 on Colab Free/Pro).

In [ ]:
import torch
import torchvision
assert torch.cuda.is_available(), 'Enable GPU: Runtime > Change runtime type > GPU'
props = torch.cuda.get_device_properties(0)
print(f'GPU: {props.name}, CC: {props.major}.{props.minor}, Memory: {props.total_memory / 1e9:.1f} GB')

## 1. CUDA NMS Kernel Source

Single-block NMS kernel optimized for Jetson Nano (Maxwell, sm_53, 128 cores, 4GB shared memory). Uses shared memory for boxes and a single-thread iterative suppression loop — ideal for small N (YOLO post-filter regime).

In [ ]:
CUDA_NMS_SOURCE = r'''
#include <torch/extension.h>
#include <vector>

// Parallel single-block NMS kernel: all 256 threads cooperate on IoU computation.
__global__ void nms_kernel(const float* __restrict__ boxes,
                           int64_t* __restrict__ keep,
                           int* __restrict__ keep_count,
                           float iou_threshold,
                           int n) {
    extern __shared__ float s_data[];
    float* s_boxes = s_data;
    int* s_suppressed = (int*)(s_data + n * 4);

    int tid = threadIdx.x;
    int blockDim_x = blockDim.x;

    for (int i = tid; i < n * 4; i += blockDim_x) s_boxes[i] = boxes[i];
    for (int i = tid; i < n; i += blockDim_x) s_suppressed[i] = 0;
    if (tid == 0) *keep_count = 0;
    __syncthreads();

    for (int i = 0; i < n; i++) {
        __shared__ bool s_is_suppressed;
        if (tid == 0) s_is_suppressed = (s_suppressed[i] != 0);
        __syncthreads();
        if (s_is_suppressed) continue;

        if (tid == 0) {
            int idx = atomicAdd(keep_count, 1);
            keep[idx] = i;
        }

        float ix1 = s_boxes[i * 4 + 0];
        float iy1 = s_boxes[i * 4 + 1];
        float ix2 = s_boxes[i * 4 + 2];
        float iy2 = s_boxes[i * 4 + 3];
        float iarea = (ix2 - ix1) * (iy2 - iy1);
        if (iarea <= 0.0f) { __syncthreads(); continue; }

        for (int j = i + 1 + tid; j < n; j += blockDim_x) {
            if (s_suppressed[j]) continue;
            float jx1 = s_boxes[j * 4 + 0];
            float jy1 = s_boxes[j * 4 + 1];
            float jx2 = s_boxes[j * 4 + 2];
            float jy2 = s_boxes[j * 4 + 3];

            float xx1 = fmaxf(ix1, jx1);
            float yy1 = fmaxf(iy1, jy1);
            float xx2 = fminf(ix2, jx2);
            float yy2 = fminf(iy2, jy2);
            float w = fmaxf(0.0f, xx2 - xx1);
            float h = fmaxf(0.0f, yy2 - yy1);
            float inter = w * h;
            float jarea = (jx2 - jx1) * (jy2 - jy1);
            float iou = inter / (iarea + jarea - inter + 1e-8f);
            if (iou > iou_threshold) s_suppressed[j] = 1;
        }
        __syncthreads();
    }
}

at::Tensor nms(at::Tensor boxes, double iou_threshold) {
    TORCH_CHECK(boxes.is_cuda(), "boxes must be a CUDA tensor");
    TORCH_CHECK(boxes.dim() == 2 && boxes.size(1) == 4, "boxes must be (N, 4)");
    TORCH_CHECK(boxes.scalar_type() == at::kFloat, "boxes must be float32");

    int n = boxes.size(0);
    auto keep = at::empty({n}, at::TensorOptions().dtype(at::kLong).device(boxes.device()));
    auto keep_count = at::zeros({1}, at::TensorOptions().dtype(at::kInt).device(boxes.device()));

    if (n == 0) return keep;

    const int MAX_N = 2300;
    TORCH_CHECK(n <= MAX_N, "nms: N exceeds shared-memory cap (", MAX_N, "); use torchvision fallback for N > ", MAX_N);

    size_t shared_mem_bytes = (size_t)n * 4 * sizeof(float) + (size_t)n * sizeof(int);
    nms_kernel<<<1, 256, shared_mem_bytes>>>(
        boxes.data_ptr<float>(),
        keep.data_ptr<int64_t>(),
        keep_count.data_ptr<int>(),
        (float)iou_threshold,
        n
    );

    int k = keep_count.item<int>();
    return keep.narrow(0, 0, k).clone();
}

PYBIND11_MODULE(TORCH_EXTENSION_NAME, m) {
    m.def("nms", &nms, "Jetson Nano NMS (CUDA)");
}
'''

## 2. Compile the Kernel

In [ ]:
from torch.utils.cpp_extension import load_inline
import shutil, os

# Clear stale build cache from previous failed attempts
cache_dir = os.path.expanduser('~/.cache/torch_extensions')
if os.path.exists(cache_dir):
    shutil.rmtree(cache_dir, ignore_errors=True)

module = load_inline(
    name='jetson_nms',
    cpp_sources=[''],
    cuda_sources=[CUDA_NMS_SOURCE],
    functions=None,  # PYBIND11_MODULE is in the CUDA source
    verbose=True,
)
print('Compilation: SUCCESS')

_MAX_KERNEL_N = 2300

def jetson_nms(boxes, scores, iou_threshold):
    order = scores.argsort(descending=True)
    sorted_boxes = boxes[order].contiguous().float()
    if sorted_boxes.shape[0] > _MAX_KERNEL_N:
        return order[torchvision.ops.nms(sorted_boxes, scores[order], iou_threshold)]
    try:
        keep_local = module.nms(sorted_boxes, float(iou_threshold))
    except Exception as e:
        print(f'  kernel failed ({e}), falling back to torchvision')
        return order[torchvision.ops.nms(sorted_boxes, scores[order], iou_threshold)]
    return order[keep_local.to(order.device)]

## 3. Correctness Tests vs torchvision

In [ ]:
test_cases = [
    ('no overlaps', [[0,0,10,10],[20,20,30,30],[50,50,60,60]], [0.9,0.8,0.7], 0.5),
    ('all overlapping', [[0,0,10,10],[0,0,10,10],[0,0,10,10]], [0.9,0.8,0.7], 0.5),
    ('partial overlap', [[0,0,10,10],[1,1,11,11],[20,20,30,30],[50,50,60,60],[2,2,12,12]], [0.9,0.8,0.7,0.6,0.5], 0.5),
    ('high threshold', [[0,0,10,10],[0,0,9,9],[0,0,8,8]], [0.9,0.8,0.7], 0.7),
    ('low threshold', [[0,0,10,10],[0,0,9,9],[0,0,8,8]], [0.9,0.8,0.7], 0.3),
    ('empty', [], [], 0.5),
    ('single box', [[0,0,10,10]], [0.9], 0.5),
]

all_passed = True
for name, box_list, score_list, iou_thres in test_cases:
    boxes = torch.tensor(box_list, dtype=torch.float32, device='cuda')
    scores = torch.tensor(score_list, dtype=torch.float32, device='cuda')
    expected = torchvision.ops.nms(boxes, scores, iou_thres)
    got = jetson_nms(boxes, scores, iou_thres)
    match = set(expected.tolist()) == set(got.tolist())
    status = 'PASS' if match else 'FAIL'
    if not match:
        all_passed = False
    print(f'  [{status}] {name}: {len(got)} kept (expected {len(expected)})')

# Random 100-box test
torch.manual_seed(42)
boxes = torch.rand(100, 4, device='cuda') * 100
boxes[:, 2:] += boxes[:, :2] + 1
scores = torch.rand(100, device='cuda')
expected = torchvision.ops.nms(boxes, scores, 0.5)
got = jetson_nms(boxes, scores, 0.5)
match = set(expected.tolist()) == set(got.tolist())
print(f'  [{"PASS" if match else "FAIL"}] 100 random boxes: {len(got)} kept (expected {len(expected)})')
all_passed = all_passed and match

print(f'\nAll correctness tests: {"PASSED" if all_passed else "FAILED"}')

## 4. Latency Benchmark

In [ ]:
import time

def benchmark(fn, boxes, scores, iou_threshold, n_warmup=5, n_runs=50):
    for _ in range(n_warmup):
        fn(boxes, scores, iou_threshold)
    torch.cuda.synchronize()
    times = []
    for _ in range(n_runs):
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        fn(boxes, scores, iou_threshold)
        torch.cuda.synchronize()
        times.append((time.perf_counter() - t0) * 1000)
    times.sort()
    return times[len(times) // 2]

print(f'{"N":>10s}  {"torchvision":>12s}  {"jetson_kernel":>14s}  {"speedup":>8s}')
print('-' * 52)
for n in [10, 50, 100, 300, 1000, 2000, 3000]:  # 3000 exceeds cap, falls back to torchvision
    torch.manual_seed(42)
    boxes = torch.rand(n, 4, device='cuda') * 640
    boxes[:, 2:] += boxes[:, :2] + 1
    scores = torch.rand(n, device='cuda')
    t_tv = benchmark(torchvision.ops.nms, boxes, scores, 0.45)
    t_jn = benchmark(jetson_nms, boxes, scores, 0.45)
    speedup = t_tv / t_jn if t_jn > 0 else 0
    print(f'N={n:<8d}  {t_tv:>10.4f}ms  {t_jn:>12.4f}ms  {speedup:>7.2f}x')

## 5. Simulate Jetson Nano (sm_53) Compilation

Cross-compile the kernel targeting sm_53 (Jetson Nano's Maxwell architecture) to verify the code is compatible. The kernel runs on the T4 but is compiled with `-gencode arch=compute_53,code=sm_53` to catch any Maxwell-incompatible instructions.

In [ ]:
# Re-compile with explicit sm_53 target to verify Jetson Nano compatibility
import os
os.environ['TORCH_CUDA_ARCH_LIST'] = '5.3'

module_sm53 = load_inline(
    name='jetson_nms_sm53',
    cpp_sources=[''],
    cuda_sources=[CUDA_NMS_SOURCE],
    functions=None,  # PYBIND11_MODULE is in the CUDA source
    verbose=True,
)
print('sm_53 (Jetson Nano) cross-compilation: SUCCESS')
print('The kernel source is compatible with Maxwell architecture.')